In [1]:
pip install regex

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install logparser3 --no-deps

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from logparser.Drain import LogParser
print("Drain working")
import pandas as pd
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

Drain working


In [5]:
log_path = "HDFS.log"
label_path = "anomaly_label.csv"
with open(log_path, "r") as f:
    for _ in range(5):
        print(f.readline())

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010

081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906

081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010

081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010

081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating



In [6]:
import os

print(os.listdir())

['.git', '.gitignore', 'anomaly_label.csv', 'HDFS.log', 'LSTM.ipynb', 'output', 'preprocessing_pipeline.ipynb', 'README.md', 'X_test.npy', 'X_train.npy', 'X_val.npy', 'y_test.npy', 'y_train.npy', 'y_val.npy']


In [7]:
log_format = '<Date> <Time> <Pid> <Level> <Component>: <Content>'

input_dir = "."          # current folder
output_dir = "./output"  # create output folder here

parser = LogParser(
    log_format=log_format,
    indir=input_dir,
    outdir=output_dir,
    depth=4,
    st=0.5
)

In [8]:
parser.parse('HDFS.log')

Parsing file: .\HDFS.log
Total lines:  11175629
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log li

In [9]:
df = pd.read_csv('./output/HDFS.log_structured.csv')
print(df.head())

   LineId   Date    Time  Pid Level                     Component  \
0       1  81109  203518  143  INFO      dfs.DataNode$DataXceiver   
1       2  81109  203518   35  INFO              dfs.FSNamesystem   
2       3  81109  203519  143  INFO      dfs.DataNode$DataXceiver   
3       4  81109  203519  145  INFO      dfs.DataNode$DataXceiver   
4       5  81109  203519  145  INFO  dfs.DataNode$PacketResponder   

                                             Content   EventId  \
0  Receiving block blk_-1608999687919862906 src: ...  09a53393   
1  BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...  3d91fa85   
2  Receiving block blk_-1608999687919862906 src: ...  09a53393   
3  Receiving block blk_-1608999687919862906 src: ...  09a53393   
4  PacketResponder 1 for block blk_-1608999687919...  d38aa58d   

                              EventTemplate  \
0    Receiving block <*> src: <*> dest: <*>   
1  BLOCK* NameSystem.allocateBlock: <*> <*>   
2    Receiving block <*> src: <*> dest: <*>   


In [10]:
def extract_block_id(text):
    match = re.search(r'(blk_-?\d+)', str(text))
    return match.group(1) if match else None

df['BlockId'] = df['Content'].apply(extract_block_id)

In [11]:
df[['Content', 'BlockId']].head()

,Content,BlockId
0,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
1,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906
2,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
3,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
4,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906


In [12]:
df = df[df['BlockId'].notnull()]
print(df[['Content', 'BlockId']].head())
print("Remaining rows:", len(df))
print("Null BlockIds:", df['BlockId'].isnull().sum())

                                             Content                   BlockId
0  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
1  BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...  blk_-1608999687919862906
2  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
3  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
4  PacketResponder 1 for block blk_-1608999687919...  blk_-1608999687919862906
Remaining rows: 11175629
Null BlockIds: 0


In [13]:
df = df.sort_values(by='LineId')
print(df[['LineId', 'BlockId']].head())

   LineId                   BlockId
0       1  blk_-1608999687919862906
1       2  blk_-1608999687919862906
2       3  blk_-1608999687919862906
3       4  blk_-1608999687919862906
4       5  blk_-1608999687919862906


In [14]:
grouped = df.groupby('BlockId')['EventId'].apply(list)
print(grouped.head())

BlockId
blk_-1000002529962039464    [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...
blk_-100000266894974466     [3d91fa85, 09a53393, 09a53393, 09a53393, 5d5de...
blk_-1000007292892887521    [09a53393, 09a53393, 3d91fa85, 09a53393, d38aa...
blk_-1000014584150379967    [09a53393, 3d91fa85, 09a53393, 09a53393, 5d5de...
blk_-1000028658773048709    [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...
Name: EventId, dtype: object


In [15]:
data = pd.DataFrame({
    'BlockId': grouped.index,
    'Sequence': grouped.values
})
print(data.head())

                    BlockId                                           Sequence
0  blk_-1000002529962039464  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...
1   blk_-100000266894974466  [3d91fa85, 09a53393, 09a53393, 09a53393, 5d5de...
2  blk_-1000007292892887521  [09a53393, 09a53393, 3d91fa85, 09a53393, d38aa...
3  blk_-1000014584150379967  [09a53393, 3d91fa85, 09a53393, 09a53393, 5d5de...
4  blk_-1000028658773048709  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...


In [16]:
labels = pd.read_csv('anomaly_label.csv')
labels['Label'] = labels['Label'].map({
    'Normal': 0,
    'Anomaly': 1
})
data = data.merge(labels, on='BlockId')
print(data.head())

                    BlockId  \
0  blk_-1000002529962039464   
1   blk_-100000266894974466   
2  blk_-1000007292892887521   
3  blk_-1000014584150379967   
4  blk_-1000028658773048709   

                                            Sequence  Label  
0  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...      0  
1  [3d91fa85, 09a53393, 09a53393, 09a53393, 5d5de...      0  
2  [09a53393, 09a53393, 3d91fa85, 09a53393, d38aa...      0  
3  [09a53393, 3d91fa85, 09a53393, 09a53393, 5d5de...      0  
4  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...      0  


In [17]:
print("Shape:", data.shape)
print("Null labels:", data['Label'].isnull().sum())

Shape: (575061, 3)
Null labels: 0


In [18]:
data = data[['BlockId', 'Sequence', 'Label']]

print(data.head())
print(data.columns)

                    BlockId  \
0  blk_-1000002529962039464   
1   blk_-100000266894974466   
2  blk_-1000007292892887521   
3  blk_-1000014584150379967   
4  blk_-1000028658773048709   

                                            Sequence  Label  
0  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...      0  
1  [3d91fa85, 09a53393, 09a53393, 09a53393, 5d5de...      0  
2  [09a53393, 09a53393, 3d91fa85, 09a53393, d38aa...      0  
3  [09a53393, 3d91fa85, 09a53393, 09a53393, 5d5de...      0  
4  [09a53393, 09a53393, 09a53393, 3d91fa85, d38aa...      0  
Index(['BlockId', 'Sequence', 'Label'], dtype='object')


In [19]:
# Get all unique EventIds
all_events = set()

for seq in data['Sequence']:
    all_events.update(seq)

# Create mapping
event2idx = {event: idx+1 for idx, event in enumerate(all_events)}

print("Number of unique events:", len(event2idx))

Number of unique events: 396


In [20]:
data['Sequence'] = data['Sequence'].apply(
    lambda seq: [event2idx[event] for event in seq]
)
print(data['Sequence'].head())
print(type(data['Sequence'].iloc[0][0]))

0    [341, 341, 341, 324, 310, 32, 310, 32, 170, 17...
1    [324, 341, 341, 341, 170, 170, 170, 310, 32, 3...
2    [341, 341, 324, 341, 310, 32, 310, 32, 310, 32...
3    [341, 324, 341, 341, 170, 170, 170, 310, 32, 3...
4    [341, 341, 341, 324, 310, 32, 310, 32, 310, 32...
Name: Sequence, dtype: object
<class 'int'>


In [21]:
from sklearn.model_selection import train_test_split

# First split: train vs temp
X_train, X_temp, y_train, y_temp = train_test_split(
    data['Sequence'],
    data['Label'],
    test_size=0.3,
    random_state=42
)

# Second split: val vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)
print(len(X_train), len(X_val), len(X_test))

402542 86259 86260


In [22]:
max_len = 50

X_train = pad_sequences(X_train, maxlen=max_len, padding='post')
X_val   = pad_sequences(X_val,   maxlen=max_len, padding='post')
X_test  = pad_sequences(X_test,  maxlen=max_len, padding='post')
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(402542, 50)
(86259, 50)
(86260, 50)


In [23]:
np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("X_test.npy", X_test)

np.save("y_train.npy", y_train)
np.save("y_val.npy", y_val)
np.save("y_test.npy", y_test)